In [ ]:
# Basis-Setup: Imports und Anzeigeoptionen
import os
import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Repo-Tabelle laden und Basis-Metriken
Dieses Notebook ist ein **simples, transformierbares Grundgerüst**.

Ablauf:
1. DB-Verbindung aus Umgebungsvariable herstellen
2. `repo`-Tabelle laden
3. Datenqualität kurz prüfen
4. Erste Metriken berechnen

In [ ]:
# 1) Build connection (Docker-Compose compatible)
# Priority: full URL from ENV -> otherwise build from individual values

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")        # docker-compose maps 5432:5432
DB_NAME = os.getenv("DB_NAME", "github_events")
DB_USER = os.getenv("DB_USER", "github")
DB_PASSWORD = os.getenv("DB_PASSWORD", "github_secret")

DATABASE_URL = os.getenv("DATABASE_URL") or (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)
print(f"DB connection ready: {DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
# 2) Daten aus Repo-Tabelle laden (auto-detect: repos oder repo)
with engine.connect() as conn:
    table_names = pd.read_sql(
        text("SELECT tablename FROM pg_catalog.pg_tables WHERE schemaname = 'public' ORDER BY tablename"),
        conn,
    )["tablename"].tolist()

if "repos" in table_names:
    source_table = "repos"
elif "repo" in table_names:
    source_table = "repo"
else:
    raise ValueError(
        "Keine Repo-Tabelle gefunden. Erwartet 'repos' oder 'repo'. "
        f"Vorhandene Tabellen: {table_names}"
    )

query = f"SELECT * FROM {source_table}"
with engine.connect() as conn:
    repo_df = pd.read_sql(text(query), conn)

print(f"Quelle: {source_table} | Geladene Zeilen: {len(repo_df):,}")
repo_df.head()

In [ ]:
# 3) Quick Data Checks
repo_df.info()

nulls = repo_df.isna().sum().sort_values(ascending=False)
nulls[nulls > 0].head(20)

In [ ]:
# 4) Basis-Measures (anpassbar an deine Spalten)
id_col = "id" if "id" in repo_df.columns else ("repo_id" if "repo_id" in repo_df.columns else None)
name_col = "name" if "name" in repo_df.columns else ("full_name" if "full_name" in repo_df.columns else None)

measures = {
    "rows": len(repo_df),
    "columns": repo_df.shape[1],
    "distinct_repo_ids": repo_df[id_col].nunique() if id_col else None,
    "distinct_repo_names": repo_df[name_col].nunique() if name_col else None,
}

pd.Series(measures, name="value")

In [ ]:
# 5) Transformations-Template (intuitiv & wiederverwendbar)
def transform_repo(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Beispiel: Standardisierte Spaltennamen
    out.columns = [c.strip().lower() for c in out.columns]

    # Beispiel: Datumsfelder robust casten (nur falls vorhanden)
    for col in ["created_at", "updated_at", "pushed_at"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce", utc=True)

    return out

repo_t = transform_repo(repo_df)
repo_t.head()

# RISING START ALGO

Bei erstem vorkommen in Events wird repo gefetched (aktueller Star stand etc geholt)


## Ansatz 1: 
## Bayesian Surprise Score für GitHub Repos

---

### Erwartete Rate

$$\hat{\lambda} = \frac{S_0}{A} \cdot w$$

### Score

$$\text{Score}_t = -\log\left(1 - F_{\text{Poisson}}(x_t - 1,\, \hat{\lambda})\right) \quad \text{if } x_t \geq n_{\min}$$

wobei $F_{\text{Poisson}}(k, \lambda) = e^{-\lambda} \sum_{i=0}^{k} \frac{\lambda^i}{i!}$

---

### Parameter & Variablen

| Symbol | Bedeutung | Herkunft |
|---|---|---|
| $x_t$ | Neu beobachtete Stars im Fenster $w$ | Live-Stream |
| $S_0$ | Stars zum Zeitpunkt des ersten Trackings | Baseline-Snapshot |
| $A$ | Geschätztes Alter des Repos in Tagen | GitHub API (`created_at`) |
| $w$ | Beobachtungsfenster in Tagen (z.B. $\frac{1}{24}$ für 1h) | Konfiguration |
| $\hat{\lambda}$ | Erwartete Stars in Fenster $w$ unter Normalverhalten | Berechnet aus $S_0, A, w$ |
| $F_{\text{Poisson}}$ | Poisson-CDF — Wahrscheinlichkeit $\leq k$ Stars zu sehen | Statistik |
| $n_{\min}$ | Noise Guard — Mindestanzahl Events für gültigen Score | Konfiguration |

---

### Interpretation

$$\text{Score}_t = -\log\underbrace{P(X \geq x_t \mid \hat{\lambda})}_{\text{p-Wert}}$$

Der Score ist der negative Logarithmus der Wahrscheinlichkeit, $x_t$ oder mehr Stars zufällig zu beobachten — gegeben das historische Wachstumstempo $\hat{\lambda}$.

| Score | Bedeutung |
|---|---|
| $< 2$ | Normales Wachstum |
| $2 - 4$ | Erhöhte Aktivität |
| $4 - 7$ | Starkes Signal |
| $> 7$ | Statistisch extrem → Hidden Gem |

---

### Sonderfall: Repo seit Erstellung getrackt

Falls kein Baseline-Snapshot, sondern vollständige Beobachtungshistorie vorhanden:

$$\hat{\lambda} = \frac{\sum_{i} x_i}{D} \cdot w$$

wobei $D$ = Anzahl beobachtete Tage seit Tracking-Start.

In [ ]:
def estimate_lambda(repo: dict, window_days: float) -> float:
    if repo["tracked_from_creation"]:
        # Full history available — use observed rate directly
        observed_days = repo["tracking_days"]
        total_stars = repo["total_observed_stars"]
        daily_rate = total_stars / max(observed_days, 1)
    else:
        # Baseline snapshot only
        daily_rate = repo["baseline_stars"] / max(repo["repo_age_days"], 1)

    return daily_rate * window_days

In [ ]:
from scipy.stats import poisson
import math

def surprise_score(
    new_stars: int,          # observed new stars in window
    baseline_stars: int,     # stars at first seen (or 0 if tracked from start)
    repo_age_days: float,    # estimated age in days (can be approximate)
    window_days: float,      # observation window size (e.g. 1/24 for 1h)
    n_min: int = 3,          # noise guard
) -> float | None:
    if new_stars < n_min:
        return None

    # Expected rate: stars per day
    daily_rate = baseline_stars / max(repo_age_days, 1)

    # Expected stars in this window
    lambda_window = daily_rate * window_days

    if lambda_window <= 0:
        # No prior history — use a weak prior (1 star/day assumed)
        lambda_window = 1.0 * window_days

    # Survival function: P(X >= x) = 1 - CDF(x-1)
    p_value = poisson.sf(new_stars - 1, mu=lambda_window)

    # Avoid log(0)
    p_value = max(p_value, 1e-10)

    return -math.log(p_value)  # Higher = more surprising

## Aufbereitung Daten

In [ ]:
with engine.connect() as conn:
    table_names = pd.read_sql(
        text("SELECT tablename FROM pg_catalog.pg_tables WHERE schemaname = 'public' ORDER BY tablename"),
        conn,
    )["tablename"].tolist()


table_names

In [ ]:

source_table = "repos"
query = f"SELECT * FROM {source_table}"
with engine.connect() as conn:
    repo_df = pd.read_sql(text(query), conn)

print(f"Quelle: {source_table} | Geladene Zeilen: {len(repo_df):,}")
repo_df

In [ ]:
repo_df["detail"].unique()

# Simple Algo: (Absolute increase during timeframe)



In [ ]:
# get for every repo the number of stars last 24h from the events
query = """
SELECT
    e.repo_id,
    r.name,
    COUNT(*) AS stars_last_24h
FROM events e
LEFT JOIN repos r ON r.repo_id = e.repo_id
WHERE e.detail = 'starred' AND e.time >= NOW() - INTERVAL '24 hours'
GROUP BY e.repo_id, r.name
ORDER BY stars_last_24h DESC
LIMIT 100
"""
with engine.connect() as conn:
    stars_24h_df = pd.read_sql(text(query), conn)
stars_24h_df

In [ ]:
# TODO



def get_repo_events(repo_id: int, event_type: str, since: pd.Timestamp) -> pd.DataFrame:

    
    pass

def get_star_increase(): # Anzahl Neue sterne während in denn letzten x days
    pass

def get_baseline_stars(): # Anzahl sterne bei der ersten beobachtung
    pass

def get_repo_age_days(): # Alter des repos in stunden
    pass    

def estimate_lambda(): # Erwartete Anzahl neuer sterne in x days
    pass